# Descoberta de Sessões

Este notebook descobre sessões de agentes a partir do AgentCore Observability para avaliação offline. Ele consulta o grupo de logs de traces do seu agente para encontrar sessões e, em seguida, salva-as em um arquivo JSON para processamento no notebook de análise.

**Dois Métodos de Descoberta:**

1. **Baseado em tempo**: Encontra todas as sessões dentro de uma janela de tempo. Use para avaliação em massa da atividade recente do agente.

2. **Baseado em pontuação**: Encontra sessões pela pontuação de avaliação existente do AgentCore. Use para reavaliar sessões com pontuação baixa com rubricas atualizadas.

**Saída:** `discovered_sessions.json` contendo IDs de sessão e metadados para o notebook de análise.

## Onde Isto se Encaixa

Este é o **Notebook 1** no fluxo de trabalho de avaliação. Após descobrir sessões aqui, você escolherá um dos dois caminhos de avaliação.

![Notebook Workflow](images/notebook_workflow.svg)

## Configuração Inicial

Importe os módulos necessários e carregue a configuração de `config.py`. Todos os valores de configuração podem ser substituídos via variáveis de ambiente antes de executar esta célula.

In [ ]:
import logging
import sys
from datetime import datetime, timedelta, timezone

sys.path.insert(0, ".")

from config import (
    AWS_REGION,
    SOURCE_LOG_GROUP,
    EVAL_RESULTS_LOG_GROUP_FULL,
    LOOKBACK_HOURS,
    MAX_SESSIONS,
    MIN_SCORE,
    MAX_SCORE,
    DISCOVERED_SESSIONS_PATH,
    EVALUATOR_NAME,
)

from utils import (
    ObservabilityClient,
    SessionDiscoveryResult,
    # SessionInfo,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

## Configuração

O `EVALUATOR_NAME` usado para descoberta baseada em pontuação é carregado de `config.py`. Ele deve corresponder ao nome do avaliador nos seus resultados de avaliação existentes. Modifique `config.py` para alterar configurações como `LOOKBACK_HOURS`, `MAX_SESSIONS` ou limites de pontuação.

In [ ]:
# EVALUATOR_NAME is loaded from config.py
print(f"Using evaluator: {EVALUATOR_NAME}")

## Inicializar Cliente

Crie o `ObservabilityClient` que gerencia as consultas do CloudWatch Logs Insights. O intervalo de tempo é calculado a partir de `LOOKBACK_HOURS` na configuração.

In [ ]:
obs_client = ObservabilityClient(
    region_name=AWS_REGION,
    log_group=SOURCE_LOG_GROUP,
)

end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(hours=LOOKBACK_HOURS)
start_time_ms = int(start_time.timestamp() * 1000)
end_time_ms = int(end_time.timestamp() * 1000)

## Descoberta Baseada em Tempo

Consulte o grupo de logs do AgentCore Observability para todos os IDs de sessão únicos dentro da janela de tempo. Retorna sessões com contagens de spans e timestamps, ordenadas pela atividade mais recente. Use este método quando quiser avaliar todas as interações recentes do agente.

In [ ]:
time_based_sessions = obs_client.discover_sessions(
    start_time_ms=start_time_ms,
    end_time_ms=end_time_ms,
    limit=MAX_SESSIONS,
)

print(f"Discovered {len(time_based_sessions)} sessions")

## Descoberta Baseada em Pontuação

Consulte o grupo de logs de resultados de avaliação do AgentCore para encontrar sessões pelas suas pontuações de avaliação existentes. Filtra sessões onde o avaliador especificado pontuou entre `MIN_SCORE` e `MAX_SCORE`. Use este método para encontrar sessões com baixo desempenho para reavaliação com rubricas atualizadas.

In [ ]:
score_based_sessions = obs_client.discover_sessions_by_score(
    evaluation_log_group=EVAL_RESULTS_LOG_GROUP_FULL,
    evaluator_name=EVALUATOR_NAME,
    start_time_ms=start_time_ms,
    end_time_ms=end_time_ms,
    min_score=MIN_SCORE,
    max_score=MAX_SCORE,
    limit=MAX_SESSIONS,
)

print(f"Discovered {len(score_based_sessions)} sessions by score")

## Selecionar Método de Descoberta

Escolha qual conjunto de sessões descobertas usar. Defina `USE_TIME_BASED = True` para resultados baseados em tempo, ou `False` para resultados baseados em pontuação. As sessões selecionadas são empacotadas em um `SessionDiscoveryResult` com metadados sobre como foram descobertas.

In [ ]:
# Set to False to use score-based discovery instead
USE_TIME_BASED = True

if USE_TIME_BASED:
    selected_sessions = time_based_sessions
    discovery_method = "time_based"
    log_group = SOURCE_LOG_GROUP
    filter_criteria = None
else:
    selected_sessions = score_based_sessions
    discovery_method = "score_based"
    log_group = EVAL_RESULTS_LOG_GROUP_FULL
    filter_criteria = {
        "evaluator_name": EVALUATOR_NAME,
        "min_score": MIN_SCORE,
        "max_score": MAX_SCORE,
    }

discovery_result = SessionDiscoveryResult(
    sessions=selected_sessions,
    discovery_time=datetime.now(timezone.utc),
    log_group=log_group,
    time_range_start=start_time,
    time_range_end=end_time,
    discovery_method=discovery_method,
    filter_criteria=filter_criteria,
)

print(f"Selected {len(selected_sessions)} sessions via {discovery_method}")

## Visualizar Sessões

Veja as primeiras 10 sessões descobertas. Para descoberta baseada em tempo, mostra o ID da sessão e a contagem de spans. Para descoberta baseada em pontuação, mostra o ID da sessão e a pontuação média de avaliação.

In [ ]:
for i, session in enumerate(selected_sessions[:10]):
    meta = session.metadata or {}
    if discovery_method == "time_based":
        print(f"{i+1}. {session.session_id} - {session.span_count} spans")
    else:
        print(f"{i+1}. {session.session_id} - avg_score: {meta.get('avg_score', 0):.2f}")

## Salvar Resultados

Salve o resultado da descoberta em JSON. Este arquivo será carregado pelo notebook de análise para processar cada sessão. O caminho de saída é configurado em `config.py` como `DISCOVERED_SESSIONS_PATH`.

In [ ]:
discovery_result.save_to_json(DISCOVERED_SESSIONS_PATH)
print(f"Saved {len(selected_sessions)} sessions to {DISCOVERED_SESSIONS_PATH}")

## Verificar Saída

Confirme que o arquivo JSON foi salvo corretamente. Após a verificação, prossiga para o notebook de análise multi-sessão para avaliar estas sessões.

In [ ]:
import json

with open(DISCOVERED_SESSIONS_PATH, "r") as f:
    saved_data = json.load(f)

print(f"Sessions: {len(saved_data['sessions'])}")
print(f"Method: {saved_data['discovery_method']}")
print(f"Time range: {saved_data['time_range_start']} to {saved_data['time_range_end']}")